In [130]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
from timm import create_model  # Vision Transformer (ViT)

In [131]:
# 🔹 Set device (force CPU for now)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Using device: {device}")


🔹 Using device: cpu


In [132]:
# 🔹 Dataset paths (Update based on your setup)
train_dir = "D:/Projects/Minor Project/Deepfake Detection/datasets/New folder/real-vs-fake/train"
val_dir = "D:/Projects/Minor Project/Deepfake Detection/datasets/New folder/real-vs-fake/valid"

In [133]:
# 🔹 Optimized transformations (Reduced image size for speed)
transform = transforms.Compose([
    transforms.Resize((224,224)),  # Reduce size for speed
    transforms.CenterCrop(224),  # Crop to 192x192
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [134]:
# 🔹 Load datasets
try:
    train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
    val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)
    print(f"✅ Train Dataset: {len(train_dataset)} images")
    print(f"✅ Validation Dataset: {len(val_dataset)} images")
except Exception as e:
    print(f"❌ Error loading datasets: {e}")


✅ Train Dataset: 100000 images
✅ Validation Dataset: 20000 images


In [135]:
# 🔹 Optimized DataLoader (More workers, larger batch size)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)


In [136]:
# 🔹 Load ViT model
model = create_model("vit_base_patch16_224", pretrained=True)
num_features = model.head.in_features
model.head = nn.Linear(num_features, 2)  # Binary classification (Real vs Fake)


In [137]:
model = model.to(device)  # Remove `.half()` for CPU (causes issues)



In [138]:
# 🔹 Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [139]:
# 🔹 Enable mixed precision training
scaler = torch.cuda.amp.GradScaler()

In [140]:
# 🔹 Training loop
num_epochs = 5
best_acc = 0.0  # Track best validation accuracy

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)  # Removed `.half()`
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(correct/total)*100:.2f}%")


Epoch 1/5:   9%|▉         | 587/6250 [1:19:04<12:42:55,  8.08s/it, acc=76.06%, loss=1.0670]


KeyboardInterrupt: 

In [ ]:
   # 🔹 Calculate training loss & accuracy
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(f"🔹 Epoch {epoch+1}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%")

    # 🔹 **Validation Loop** 🔹
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():  # Faster validation
        for val_inputs, val_labels in val_loader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
            val_outputs = model(val_inputs)
            _, val_preds = torch.max(val_outputs, 1)
            val_correct += torch.sum(val_preds == val_labels).item()
            val_total += val_labels.size(0)

 

In [ ]:
    val_acc = (val_correct / val_total) * 100
    print(f"✅ Validation Accuracy: {val_acc:.2f}%")

    # 🔹 Save best model based on validation accuracy
    if val_acc > best_acc:
        best_acc = val_acc
        model_path = "D:/Projects/Minor Project/Deepfake Detection/best_vit_model.pth"
        torch.save(model.state_dict(), model_path)
        print(f"🎉 Best model saved at epoch {epoch+1} with validation accuracy {best_acc:.2f}%")

print("✅ Training complete!")